# 03 - Benchmarking Models and Selecting Production Candidate

## Definition
Benchmarking compares multiple models under same split and metrics.

## Theory
Production selection should optimize tradeoff: quality + latency + maintainability.

## Motivation
Do not choose model by intuition. Use reproducible ranking with clear criteria.


In [1]:
from pathlib import Path
import os

CWD = Path.cwd()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/benchmarks").mkdir(parents=True, exist_ok=True)


In [2]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

bench_csv = Path("outputs/benchmarks/model_benchmark.csv")
automl_json = Path("outputs/benchmarks/automl_benchmark.json")

if not bench_csv.exists() or not automl_json.exists():
    subprocess.run([sys.executable, "scripts/train_model.py"], check=True)

benchmark_df = pd.read_csv(bench_csv)
benchmark_df.sort_values(["f1_macro", "accuracy"], ascending=False)


,model_name,fit_time_ms,predict_time_ms,supports_proba,status,error,accuracy,precision_macro,recall_macro,f1_macro,roc_auc_ovr
5,KNN,0.364,4.138,True,ok,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
0,LogisticRegression,6.663,2.876,True,ok,NaN,0.966667,0.969697,0.966667,0.966583,1.000000
4,SVM,1.340,2.787,True,ok,NaN,0.966667,0.969697,0.966667,0.966583,0.996667
8,CatBoost,110.340,4.120,True,ok,NaN,0.966667,0.969697,0.966667,0.966583,0.990000
1,DecisionTree,0.602,2.562,True,ok,NaN,0.933333,0.933333,0.933333,0.933333,0.950000
3,ExtraTrees,89.049,10.958,True,ok,NaN,0.933333,0.933333,0.933333,0.933333,0.993333
6,XGBoost,314.177,29.611,True,ok,NaN,0.933333,0.933333,0.933333,0.933333,0.971667
2,RandomForest,127.302,10.981,True,ok,NaN,0.900000,0.902357,0.900000,0.899749,0.986667
7,LightGBM,33.004,4.459,True,ok,NaN,0.900000,0.902357,0.900000,0.899749,0.963333


## Required Model Families
This table includes:
- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost
- LightGBM
- CatBoost
- SVM
- KNN

Optional libraries load when installed.


In [3]:
ranked = benchmark_df[benchmark_df["status"] == "ok"].sort_values(["f1_macro", "accuracy"], ascending=False)
ranked[["model_name", "accuracy", "precision_macro", "recall_macro", "f1_macro", "predict_time_ms"]]


,model_name,accuracy,precision_macro,recall_macro,f1_macro,predict_time_ms
5,KNN,1.000000,1.000000,1.000000,1.000000,4.138
0,LogisticRegression,0.966667,0.969697,0.966667,0.966583,2.876
4,SVM,0.966667,0.969697,0.966667,0.966583,2.787
8,CatBoost,0.966667,0.969697,0.966667,0.966583,4.120
1,DecisionTree,0.933333,0.933333,0.933333,0.933333,2.562
3,ExtraTrees,0.933333,0.933333,0.933333,0.933333,10.958
6,XGBoost,0.933333,0.933333,0.933333,0.933333,29.611
2,RandomForest,0.900000,0.902357,0.900000,0.899749,10.981
7,LightGBM,0.900000,0.902357,0.900000,0.899749,4.459


In [4]:
automl_df = pd.read_json("outputs/benchmarks/automl_benchmark.json")
automl_df


,framework,status,best_model,metric_name,metric_value,notes
0,LazyPredict,ok,LinearDiscriminantAnalysis,F1 Score,1.000000,Auto baseline from LazyPredict candidate search
1,FLAML,ok,catboost,accuracy,0.966667,Time-budgeted AutoML search
2,PyCaret,skipped,None,None,NaN,"('Pycaret only supports python 3.9, 3.10, 3.11..."


## LazyPredict vs FLAML vs PyCaret

### Why Each Exists
- **LazyPredict**: rapid baseline sweep
- **FLAML**: efficient AutoML search with time budget
- **PyCaret**: low-code experiment management and model comparison

### Strengths
- LazyPredict: speed and simplicity
- FLAML: strong cost-performance efficiency
- PyCaret: rich experiment abstraction

### Weaknesses
- LazyPredict: less control and deeper tuning
- FLAML: less pedagogical transparency than manual modeling
- PyCaret: heavier dependency stack

### Tradeoff
Use all three for teaching breadth, then pick production path based on control vs speed.


In [5]:
bench_path = Path("outputs/benchmarks/model_benchmark_from_notebook03.csv")
ranked.to_csv(bench_path, index=False)
print(f"Saved: {bench_path}")

automl_path = Path("outputs/benchmarks/automl_benchmark_from_notebook03.json")
automl_path.write_text(automl_df.to_json(orient="records", indent=2), encoding="utf-8")
print(f"Saved: {automl_path}")


Saved: outputs/benchmarks/model_benchmark_from_notebook03.csv
Saved: outputs/benchmarks/automl_benchmark_from_notebook03.json
